In [ ]:
## Test GPU
import torch

def get_device():
    """Get the appropriate device for model inference."""
    if torch.cuda.is_available():
        return "cuda"
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"
    return "cpu"

DEVICE = get_device()

print(f"Using device: {DEVICE}")

Using device: cuda


## create the dataset in the db

```
python launch_remote.py --root /share/home/e2406743/dataset --NCcsv /share/home/e2406743/dataset/dataset/NC/dugong_environmental_variables_NC.csv --WPcsv /share/home/e2406743/dataset/dataset/WP/dugong_environmental_variables_WP.xlsx --launch false --name_dataset dugong --path_database /share/home/e2406743/fiftyone/mongodb --port_database 44123
```
## create manually the mongodb service

```bash
# 1. Kill any old "ghost" processes
pkill -u $(whoami) -f mongod

# 2. Start the database manually
 # Use your /tmp path for speed, or /share/projects/... for persistence
mongod --dbpath /share/home/e2406743/fiftyone/mongodb --port 44123 --fork --logpath /share/home/e2406743/fiftyone/mongodb/mongo.log
 # OR IF IN TEMP FOLDER 
mongod --dbpath /tmp/e2406743/mongodb --port 44123 --logpath /tmp/e2406743/mongodb/mongo.log --fork
```

/share/home/e2406734/fiftyone/mongodb 

In [1]:
import os

# Define the URI to point to your manual process
os.environ["FIFTYONE_DATABASE_URI"] = "mongodb://localhost:44123"

import fiftyone as fo

# Verify connection
print(fo.core.odm.database.get_db_conn()) 
# STICK to localhost:44123 

You are running the oldest supported major version of MongoDB. Please refer to https://deprecation.voxel51.com for deprecation notices. You can suppress this exception by setting your `database_validation` config parameter to `False`. See https://docs.voxel51.com/user_guide/config.html#configuring-a-mongodb-connection for more information
Database(MongoClient(host=['localhost:44123'], document_class=dict, tz_aware=False, connect=True, appname='fiftyone'), 'fiftyone')


In [2]:
fo.list_datasets()

['dugong']

In [3]:
from dotenv import load_dotenv
load_dotenv()

import fiftyone as fo
import fiftyone.utils.torch as fout
from huggingface_hub import login

if "HUGGING_FACE_API" in os.environ:
    login(token=os.environ["HUGGING_FACE_API"])

In [ ]:
client = fo.core.odm.database.get_db_client()
print(client)  # Should show localhost:44123

MongoClient(host=['localhost:44123'], document_class=dict, tz_aware=False, connect=True, appname='fiftyone')


In [6]:
## if need to create a new dataset
## python launch_remote.py --root /share/home/e2406743/dataset --NCcsv /share/home/e2406743/dataset/dataset/NC/dugong_environmental_variables_NC.csv --WPcsv /share/home/e2406743/dataset/dataset/WP/dugong_environmental_variables_WP.xlsx --launch false --name_dataset dugong --path_database /tmp/e2406734/mongodb --port_database 44123

In [7]:
print(fo.config)

{
    "allow_legacy_orchestrators": false,
    "batcher_static_size": 100,
    "batcher_target_latency": 0.2,
    "batcher_target_size_bytes": 1048576,
    "database_admin": true,
    "database_compressor": null,
    "database_dir": "/share/home/e2406743/.fiftyone/var/lib/mongo",
    "database_name": "fiftyone",
    "database_uri": "mongodb://localhost:44123",
    "database_validation": true,
    "dataset_zoo_dir": "/share/home/e2406743/fiftyone",
    "dataset_zoo_manifest_paths": null,
    "default_app_address": "localhost",
    "default_app_port": 5151,
    "default_batch_size": null,
    "default_batcher": "latency",
    "default_dataset_dir": "/share/home/e2406743/fiftyone",
    "default_image_ext": ".jpg",
    "default_ml_backend": "torch",
    "default_parallelization_method": null,
    "default_process_pool_workers": null,
    "default_sequence_idx": "%06d",
    "default_thread_pool_workers": null,
    "default_video_ext": ".mp4",
    "delegated_operation_monitor_interval": 60,


In [8]:
import fiftyone.zoo as foz
import fiftyone.brain as fob
from fiftyone import ViewField as F
import fiftyone.operators as foo
from fiftyone.utils.huggingface import load_from_hub
import cv2
import numpy as np
import fiftyone.brain as fob
import fiftyone.utils.transformers as fout

In [9]:
    print(fo.core.odm.database.get_db_conn())
    print(fo.core.odm.database.get_db_client())

Database(MongoClient(host=['localhost:44123'], document_class=dict, tz_aware=False, connect=True, appname='fiftyone'), 'fiftyone')
MongoClient(host=['localhost:44123'], document_class=dict, tz_aware=False, connect=True, appname='fiftyone')


In [4]:
dataset = fo.load_dataset("dugong")

## load the views
nc_view = dataset.load_saved_view("New_Caledonia")
wp_view = dataset.load_saved_view("West_Papua")

In [6]:
next(iter(dataset))

<Sample: {
    'id': '69add898cfd942e1c6b5d3ad',
    'media_type': 'image',
    'filepath': '/share/home/e2406743/dataset/dataset/NC/Flight_226/images/GH034226-619fa2d56d4d3_92.jpeg',
    'tags': ['Flight_226', 'NC', 'NC'],
    'metadata': <ImageMetadata: {
        'size_bytes': 712381,
        'mime_type': 'image/jpeg',
        'width': 2704,
        'height': 1520,
        'num_channels': 3,
    }>,
    'created_at': datetime.datetime(2026, 3, 8, 20, 14, 16, 713000),
    'last_modified_at': datetime.datetime(2026, 3, 9, 8, 53, 5, 136000),
    'region': 'NC',
    'subregion': 'NC',
    'mission_name': 'Flight_226',
    'sea_state': 0,
    'turbidity_global': 1,
    'turbidity_local': 'no',
    'sun_glitter': '0-0',
    'cloud_reflection': '0-0',
    'habitat_type': 'coral',
    'background_complexity': 'high',
    'coral': 'P',
    'sand': 'A',
    'dense_seagrass': 'A',
    'open_sea': 'P',
    'sparse_seagrass': 'A',
    'ground_truth': <Detections: {
        'detections': [
       

In [11]:
## Session launch

In [5]:
# On a cluster: auto=False prevents it trying to open a browser
# Use port forwarding: ssh -L 5151:localhost:5151 user@cluster
session = fo.launch_app(dataset,
                        port=5151,
                        auto=False)
print(session.url)  

Session launched. Run `session.show()` to open the App in a cell output.
http://localhost:5151/


## Model & Processor

In [11]:
from transformers import AutoImageProcessor, AutoModel
import fiftyone.utils.transformers as fout

processor = AutoImageProcessor.from_pretrained("facebook/dinov3-vitl16-pretrain-lvd1689m")
base_model = AutoModel.from_pretrained("facebook/dinov3-vitl16-pretrain-lvd1689m")

# Convert using the transformers utility
# defaults to feature extraction
model = fout.convert_transformers_model(base_model, image_processor=processor)

Loading weights:   0%|          | 0/415 [00:00<?, ?it/s]

In [12]:
print(model.transforms)

In [13]:
print(model)

## Patch Embeddings

In [23]:
import fiftyone.core.models as fom

class DinoV3PatchModel(fom.Model):

    def __init__(self, model, processor, device=None):
        self.model = model
        self.processor = processor
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.model.eval()

    @property
    def media_type(self):
        return "image"

    @property
    def has_logits(self):
        return False

    @property
    def has_embeddings(self):
        return True

    @property
    def ragged_batches(self):
        return False

    @property
    def transforms(self):
        return None  # Let embed() handle everything

    def _preprocess(self, img):
        """Convert any input format to a processor-ready PIL image."""
        from PIL import Image
        import numpy as np
        if isinstance(img, torch.Tensor):
            img = img.cpu().numpy()
        if isinstance(img, np.ndarray):
            if img.dtype != np.uint8:
                img = (img * 255).clip(0, 255).astype(np.uint8)
            img = Image.fromarray(img)
        if not isinstance(img, Image.Image):
            img = Image.fromarray(img)
        return img.convert("RGB")

    def embed(self, img):
        pil = self._preprocess(img)
        inputs = self.processor(images=pil, return_tensors="pt")
        pixel_values = inputs["pixel_values"].to(self.device)  # (1, 3, H, W)
        with torch.inference_mode():
            outputs = self.model(pixel_values=pixel_values)
        return outputs.pooler_output.squeeze(0).cpu().numpy()

    def embed_all(self, imgs):
        from PIL import Image
        pil_imgs = [self._preprocess(img) for img in imgs]
        inputs = self.processor(images=pil_imgs, return_tensors="pt")
        pixel_values = inputs["pixel_values"].to(self.device)  # (B, 3, H, W)
        with torch.inference_mode():
            outputs = self.model(pixel_values=pixel_values)
        return outputs.pooler_output.cpu().numpy()


model = DinoV3PatchModel(base_model, processor)
# Run
patch_embeddings = dataset.compute_patch_embeddings(
    model,
    patches_field="ground_truth",
    embeddings_field="patch_embeddings",
    num_workers=0,
    progress=True,
)

Ignoring unsupported `num_workers` parameter
 100% |███████████████| 2755/2755 [3.4m elapsed, 0s remaining, 10.1 samples/s]      


In [12]:
patch_results = fob.compute_visualization(
    dataset,
    patches_field="ground_truth",
    embeddings="patch_embeddings",
    method="tsne",
    brain_key="patch_emb_dinov3_tsne",
    create_index=True,   
    seed=42,
    max_iter = 2000, 
    progress=True,
)

Generating visualization...
[t-SNE] Computing 91 nearest neighbors...
[t-SNE] Indexed 7161 samples in 0.001s...
[t-SNE] Computed neighbors for 7161 samples in 0.304s...
[t-SNE] Computed conditional probabilities for sample 1000 / 7161
[t-SNE] Computed conditional probabilities for sample 2000 / 7161
[t-SNE] Computed conditional probabilities for sample 3000 / 7161
[t-SNE] Computed conditional probabilities for sample 4000 / 7161
[t-SNE] Computed conditional probabilities for sample 5000 / 7161
[t-SNE] Computed conditional probabilities for sample 6000 / 7161
[t-SNE] Computed conditional probabilities for sample 7000 / 7161
[t-SNE] Computed conditional probabilities for sample 7161 / 7161
[t-SNE] Mean sigma: 2.444409
[t-SNE] Computed conditional probabilities in 0.130s
[t-SNE] Iteration 50: error = 85.2473755, gradient norm = 0.0108855 (50 iterations in 0.452s)
[t-SNE] Iteration 100: error = 82.8385010, gradient norm = 0.0008517 (50 iterations in 0.370s)
[t-SNE] Iteration 150: error = 8

In [16]:
## plot results 
# Visualize embeddings, colored by ground truth label
plot = patch_results.visualize(labels="ground_truth.detections.subregion")
plot.show(height=720)

session.plots.attach(plot)

FigureWidget({
    'data': [{'customdata': array(['69add897cfd942e1c6b5c976', '69add897cfd942e1c6b5c977',
                                   '69add897cfd942e1c6b5c978', ..., '69add897cfd942e1c6b5ce8e',
                                   '69add897cfd942e1c6b5ce8f', '69add897cfd942e1c6b5ce90'],
                                  shape=(1307,), dtype=object),
              'hovertemplate': ('<b>subregion: %{text}</b><br>x' ... ': %{customdata}<extra></extra>'),
              'line': {'color': '#3366CC'},
              'mode': 'markers',
              'name': np.str_('FRIWEN'),
              'showlegend': True,
              'text': array(['FRIWEN', 'FRIWEN', 'FRIWEN', ..., 'FRIWEN', 'FRIWEN', 'FRIWEN'],
                            shape=(1307,), dtype=object),
              'type': 'scattergl',
              'uid': '5cda597e-4e28-4bef-a6e7-f9c5085933bf',
              'x': {'bdata': ('PmUcwaA2/UH/XABCnwT+Qa1nD0IENQ' ... '1C4nohQklq9UF69axBIir5QfRq+UE='),
                    'dtype': 'f4'},

In [11]:
session.open_tab()

<IPython.core.display.Javascript object>

In [12]:
session.close()

In [12]:
next(iter(dataset))

<Sample: {
    'id': '69add898cfd942e1c6b5d3ad',
    'media_type': 'image',
    'filepath': '/share/home/e2406743/dataset/dataset/NC/Flight_226/images/GH034226-619fa2d56d4d3_92.jpeg',
    'tags': ['Flight_226', 'NC', 'NC'],
    'metadata': <ImageMetadata: {
        'size_bytes': 712381,
        'mime_type': 'image/jpeg',
        'width': 2704,
        'height': 1520,
        'num_channels': 3,
    }>,
    'created_at': datetime.datetime(2026, 3, 8, 20, 14, 16, 713000),
    'last_modified_at': datetime.datetime(2026, 3, 9, 9, 22, 52, 329000),
    'region': 'NC',
    'subregion': 'NC',
    'mission_name': 'Flight_226',
    'sea_state': 0,
    'turbidity_global': 1,
    'turbidity_local': 'no',
    'sun_glitter': '0-0',
    'cloud_reflection': '0-0',
    'habitat_type': 'coral',
    'background_complexity': 'high',
    'coral': 'P',
    'sand': 'A',
    'dense_seagrass': 'A',
    'open_sea': 'P',
    'sparse_seagrass': 'A',
    'ground_truth': <Detections: {
        'detections': [
      

## Full Embeddings

In [14]:
embeddings = dataset.compute_embeddings(
    model,
    embeddings_field='full_embeddings',
    num_workers =0,
    progress=True,  
)

 100% |███████████████| 2755/2755 [4.6m elapsed, 0s remaining, 6.3 samples/s]       


In [14]:
## compute visualization 
full_emb_results = fob.compute_visualization(
    dataset,
    embeddings="full_embeddings",
    method="tsne",
    brain_key="full_emb_tsne_dinov3",
    create_index=True,   
    seed=42,
    progress=True,
    max_iters = 1750
)

Generating visualization...
[t-SNE] Computing 91 nearest neighbors...
[t-SNE] Indexed 2755 samples in 0.000s...
[t-SNE] Computed neighbors for 2755 samples in 0.204s...
[t-SNE] Computed conditional probabilities for sample 1000 / 2755
[t-SNE] Computed conditional probabilities for sample 2000 / 2755
[t-SNE] Computed conditional probabilities for sample 2755 / 2755
[t-SNE] Mean sigma: 1.641154
[t-SNE] Computed conditional probabilities in 0.048s
[t-SNE] Iteration 50: error = 68.1386948, gradient norm = 0.0356388 (50 iterations in 0.246s)
[t-SNE] Iteration 100: error = 61.7722702, gradient norm = 0.0131675 (50 iterations in 0.147s)
[t-SNE] Iteration 150: error = 59.9292450, gradient norm = 0.0080696 (50 iterations in 0.138s)
[t-SNE] Iteration 200: error = 58.9865456, gradient norm = 0.0065848 (50 iterations in 0.138s)
[t-SNE] Iteration 250: error = 58.4066734, gradient norm = 0.0047169 (50 iterations in 0.138s)
[t-SNE] KL divergence after 250 iterations with early exaggeration: 58.406673

In [15]:
## plot results 
# Visualize embeddings, colored by ground truth label
plot = full_emb_results.visualize(labels="subregion")
plot.show(height=720)

session.plots.attach(plot)

FigureWidget({
    'data': [{'customdata': array(['69add89acfd942e1c6b5d87c', '69add89acfd942e1c6b5d87d',
                                   '69add89acfd942e1c6b5d87e', ..., '69add89acfd942e1c6b5db84',
                                   '69add89acfd942e1c6b5db85', '69add89acfd942e1c6b5db86'],
                                  shape=(779,), dtype=object),
              'hovertemplate': ('<b>subregion: %{text}</b><br>x' ... ': %{customdata}<extra></extra>'),
              'line': {'color': '#3366CC'},
              'mode': 'markers',
              'name': np.str_('FRIWEN'),
              'showlegend': True,
              'text': array(['FRIWEN', 'FRIWEN', 'FRIWEN', ..., 'FRIWEN', 'FRIWEN', 'FRIWEN'],
                            shape=(779,), dtype=object),
              'type': 'scattergl',
              'uid': 'fb2634da-78e7-4160-ab9f-3e55771b8047',
              'x': {'bdata': ('MYaAQBP10kB8nLVAI5XnQADV8EDy3W' ... 'zB5Xczwfw9f0FBp3tBjQOAQXvzfkE='),
                    'dtype': 'f4'},
 

In [29]:
# Plot inline (if in Jupyter with display support)
plot = results.visualize(labels="ground_truth.detections.region")
plot.show()

FigureWidget({
    'data': [{'customdata': array(['69add895cfd942e1c6b5b7b4', '69add895cfd942e1c6b5b7b5',
                                   '69add895cfd942e1c6b5b7b6', ..., '69add896cfd942e1c6b5c58b',
                                   '69add896cfd942e1c6b5c58c', '69add896cfd942e1c6b5c58d'],
                                  shape=(3546,), dtype=object),
              'hovertemplate': ('<b>region: %{text}</b><br>x, y' ... ': %{customdata}<extra></extra>'),
              'line': {'color': '#3366CC'},
              'mode': 'markers',
              'name': np.str_('NC'),
              'showlegend': True,
              'text': array(['NC', 'NC', 'NC', ..., 'NC', 'NC', 'NC'], shape=(3546,), dtype=object),
              'type': 'scattergl',
              'uid': '6f06bd37-0622-4c18-8a0f-2ed2b9234a3d',
              'x': {'bdata': ('+ejswdzZQsHeZEPBmexCwRFzQ8EKVN' ... 'yFwVTuwsGU9bZBhXUTwguipEGGYw5C'),
                    'dtype': 'f4'},
              'y': {'bdata': ('K/ccQmxtnEG9jJtBeQydQVMG

In [31]:
next(iter(dataset))

<Sample: {
    'id': '69add898cfd942e1c6b5d3ad',
    'media_type': 'image',
    'filepath': '/share/home/e2406743/dataset/dataset/NC/Flight_226/images/GH034226-619fa2d56d4d3_92.jpeg',
    'tags': ['Flight_226', 'NC', 'NC'],
    'metadata': <ImageMetadata: {
        'size_bytes': 712381,
        'mime_type': 'image/jpeg',
        'width': 2704,
        'height': 1520,
        'num_channels': 3,
    }>,
    'created_at': datetime.datetime(2026, 3, 8, 20, 14, 16, 713000),
    'last_modified_at': datetime.datetime(2026, 3, 8, 20, 51, 20, 872683),
    'region': 'NC',
    'subregion': 'NC',
    'mission_name': 'Flight_226',
    'sea_state': 0,
    'turbidity_global': 1,
    'turbidity_local': 'no',
    'sun_glitter': '0-0',
    'cloud_reflection': '0-0',
    'habitat_type': 'coral',
    'background_complexity': 'high',
    'coral': 'P',
    'sand': 'A',
    'dense_seagrass': 'A',
    'open_sea': 'P',
    'sparse_seagrass': 'A',
    'ground_truth': <Detections: {
        'detections': [
     

In [32]:
fo.close_app()

In [28]:
for sample in dataset.iter_samples(autosave=True, progress=True):
    region = sample["region"]
    if sample.ground_truth and sample.ground_truth.detections:
        for det in sample.ground_truth.detections:
            det["region"] = region

 100% |███████████████| 2755/2755 [7.7s elapsed, 0s remaining, 623.9 samples/s]       


In [ ]:
# DINOv3
from transformers import Dinov2ForImageClassification
model = Dinov2ForImageClassification.from_pretrained(
    "facebook/dinov2-small-imagenet1k-1-layer"
)

Loading weights:   0%|          | 0/225 [00:00<?, ?it/s]

In [ ]:
print(model)

Dinov2ForImageClassification(
  (dinov2): Dinov2Model(
    (embeddings): Dinov2Embeddings(
      (patch_embeddings): Dinov2PatchEmbeddings(
        (projection): Conv2d(3, 384, kernel_size=(14, 14), stride=(14, 14))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): Dinov2Encoder(
      (layer): ModuleList(
        (0-11): 12 x Dinov2Layer(
          (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
          (attention): Dinov2Attention(
            (attention): Dinov2SelfAttention(
              (query): Linear(in_features=384, out_features=384, bias=True)
              (key): Linear(in_features=384, out_features=384, bias=True)
              (value): Linear(in_features=384, out_features=384, bias=True)
            )
            (output): Dinov2SelfOutput(
              (dense): Linear(in_features=384, out_features=384, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (layer_scale1): Dinov2Laye

## Compute Embeddings

In [ ]:
# compute embeedings for the given model
embeddings  = dataset.compute_embeddings(model,
                                        batch_size=256,
                                        num_workers=0,
                                        embeddings_field="embeddings",
                                        progress=True)



 100% |███████████████| 2755/2755 [4.7m elapsed, 0s remaining, 8.8 samples/s]     


### save embeddings back in the shared folder 

In [ ]:
folder_root = "/share/projects/embeddings/"
model_emb ="dinov2"
name_emb = f"full_embeddings.npy"

## create dir in case doesnt exist
os.makedirs(os.path.join(folder_root, model_emb), exist_ok=True)

## save embeddings back in the share storage
np.save(os.path.join(folder_root, model_emb, name_emb), embeddings)

print("Embeddings saved to shared storage!")

In [ ]:
results = fob.compute_visualization(
     dataset,
     embeddings=embeddings,
     method="tsne",
     brain_key="dino_viz",
  ##  pre_pca=50, # Reduces dimensions to 50 before running t-SNE
 )
